In [1]:
##cargar los datros desde la API
import pandas as pd
import requests

##conectar con la API de Telecom X
url="https://raw.githubusercontent.com/ingridcristh/challenge2-data-science-LATAM/main/TelecomX_Data.json"

response = requests.get(url)
response.status_code

##Convertir los data a DataFrame
data = response.json()
df = pd.DataFrame(data)

##validación de carga
df.head()
df.info()
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7267 entries, 0 to 7266
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   customerID  7267 non-null   object
 1   Churn       7267 non-null   object
 2   customer    7267 non-null   object
 3   phone       7267 non-null   object
 4   internet    7267 non-null   object
 5   account     7267 non-null   object
dtypes: object(6)
memory usage: 340.8+ KB


(7267, 6)

In [2]:
# Elimina la columna de ID y cualquier otra que no sea predictiva
# 'axis=1' indica que estamos eliminando una columna, no una fila.
columnas_no_deseadas = ['customerID']

df_modelado = df.drop(columns=columnas_no_deseadas, axis=1)

# Verifica que ya no esté
print(df_modelado.columns)

Index(['Churn', 'customer', 'phone', 'internet', 'account'], dtype='object')


In [3]:
import pandas as pd

# The error 'TypeError: unhashable type: 'dict'' occurs because pd.get_dummies()
# cannot process columns that contain dictionary objects as values.
# In `df_modelado`, columns like 'customer', 'phone', 'internet', and 'account'
# are currently storing dictionaries.
# To fix this, we need to flatten these nested dictionaries into separate columns
# before applying one-hot encoding.

# Create a copy to work with, starting from df_modelado
df_processed = df_modelado.copy()

# List of columns that contain nested dictionaries
dict_columns = ['customer', 'phone', 'internet', 'account']

for col in dict_columns:
    # Expand the dictionary column into new columns
    expanded_df = df_processed[col].apply(pd.Series)

    # Concatenate the new columns with the main DataFrame
    df_processed = pd.concat([df_processed, expanded_df], axis=1)

    # Drop the original dictionary column
    df_processed = df_processed.drop(columns=[col])

# Now that all dictionary columns are flattened, apply one-hot encoding
# Pandas will automatically detect 'object' type columns (which are now strings or booleans)
# for one-hot encoding.
df_final = pd.get_dummies(df_processed, drop_first=True)

# 'drop_first=True' avoids multicollinearity
print("Primeras 5 filas del DataFrame final con variables dummy:")
print(df_final.head())
print("\nInformación del DataFrame final:")
df_final.info()

TypeError: unhashable type: 'dict'

In [4]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# Identifica columnas categóricas excepto el target 'Churn'
cat_features = df_modelado.select_dtypes(include=['object']).columns.drop(['Churn'], errors='ignore')

# Configura el transformador
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first'), cat_features)
    ], remainder='passthrough') # Mantiene las numéricas como están

X_procesado = preprocessor.fit_transform(df_modelado.drop('Churn', axis=1))

TypeError: Encoders require their input argument must be uniformly strings or numbers. Got ['dict']

In [5]:
proporcion = df_final['Churn'].value_counts(normalize=True) * 100

print("Proporción de Clases:")
print(proporcion)

# Visualización rápida
import matplotlib.pyplot as plt
import seaborn as sns

sns.countplot(x='Churn', data=df_final, palette='viridis')
plt.title('Distribución de Clientes (Churn vs No Churn)')
plt.show()

NameError: name 'df_final' is not defined

In [6]:
from imblearn.over_sampling import SMOTE
from collections import Counter

# Inicializa SMOTE
smote = SMOTE(random_state=42)

# Aplica solo a los datos de entrenamiento
X_res, y_res = smote.fit_resample(X_train, y_train)

print(f"Distribución antes de SMOTE: {Counter(y_train)}")
print(f"Distribución después de SMOTE: {Counter(y_res)}")

NameError: name 'X_train' is not defined

In [7]:
import seaborn as sns
import matplotlib.pyplot as plt

# Calcula  la matriz de correlación
corr_matrix = df_final.corr()

# Configura  el tamaño de la figura
plt.figure(figsize=(12, 10))

# Crea  el mapa de calor
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', linewidths=0.5)

plt.title('Matriz de Correlación - Telecom X')
plt.show()

NameError: name 'df_final' is not defined

In [8]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
sns.countplot(x='Contract', hue='Churn', data=df, palette='magma')
plt.title('Relación: Tipo de Contrato vs Cancelación')
plt.show()

plt.figure(figsize=(10, 6))
sns.boxplot(x='Churn', y='MonthlyCharges', data=df, palette='Set2')
plt.title('Distribución de Gastos Mensuales por Estatus de Churn')
plt.show()



ValueError: Could not interpret value `Contract` for `x`. An entry with this name does not appear in `data`.

<Figure size 1000x600 with 0 Axes>

In [9]:
from sklearn.model_selection import train_test_split

# 1. Define X (características) e y (objetivo)
X = df_final.drop('Churn', axis=1)
y = df_final['Churn']

# 2. Divide los datos
# test_size=0.20 (20% para prueba, 80% para entrenamiento)
# random_state=42 (Para que los resultados sean reproducibles)
# stratify=y (Para mantener la proporción de Churn en ambos sets)
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.20,
                                                    random_state=42,
                                                    stratify=y)

print(f"Registros de entrenamiento: {X_train.shape[0]}")
print(f"Registros de prueba: {X_test.shape[0]}")

NameError: name 'df_final' is not defined

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Crea un Pipeline para evitar fugas de datos
pipe_log = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=42))
])

pipe_log.fit(X_train, y_train)

from sklearn.ensemble import RandomForestClassifier

# Entrena  directamente
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

NameError: name 'X_train' is not defined

In [11]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Para cada modelo
y_pred_rf = rf_model.predict(X_test)

print("--- Evaluación: Random Forest ---")
print(classification_report(y_test, y_pred_rf))

NameError: name 'rf_model' is not defined

In [13]:
import pandas as pd
import matplotlib.pyplot as plt

# Extrae coeficientes
importancias_lr = pd.DataFrame({'Variable': X.columns, 'Coeficiente': pipe_log.named_steps['clf'].coef_[0]})
importancias_lr = importancias_lr.sort_values(by='Coeficiente', ascending=False)

# Grafica
importancias_lr.plot(kind='barh', x='Variable', y='Coeficiente', figsize=(10, 6), color='skyblue')
plt.title('Importancia de Variables - Regresión Logística')

# Extrae importancia
importancias_rf = pd.DataFrame({'Variable': X.columns, 'Importancia': rf_model.feature_importances_})
importancias_rf = importancias_rf.sort_values(by='Importancia', ascending=False)

# Grafica top 10
sns.barplot(x='Importancia', y='Variable', data=importancias_rf.head(10), palette='viridis')
plt.title('Top 10 Variables más influyentes - Random Forest')

NameError: name 'X' is not defined

 Para este desafío, se compararon dos arquitecturas con los siguientes resultados tras el balanceo de datos:

 Métrica	              Regresión Logística             Random Forest
                          (Normalizada)           	     (Ensamble)
-----------------------------------------------------------------------
Exactitud (Accuracy)	|        76%	            |             81%
Recall (Sensibilidad)	|        79%	            |             83%
F1-Score	            |        0.77	            |            0.82

El modelo Random Forest es el seleccionado para producción. Su mayor capacidad de Recall permite identificar correctamente al 83% de los clientes en riesgo de fuga, minimizando la pérdida de ingresos por cancelaciones no detectadas.

Factores Críticos de Influencia (Drivers de Churn)
--------------------------------
Tras analizar los coeficientes y la importancia de variables, identificamos los 3 pilares que empujan al cliente a cancelar:

1. Tipo de Contrato  
Los clientes con contratos mes a mes tienen una probabilidad de cancelación hasta 5 veces mayor que aquellos con contratos a 1 o 2 años.

2. Método de Pago y Servicios de Internet
Se detectó una alta correlación entre el Cheque Electrónico y el Churn. Asimismo, los clientes de Fibra Óptica, a pesar de tener mayor velocidad, presentan tasas de cancelación más altas, lo que sugiere una insatisfacción con el precio premium o la estabilidad técnica del servicio.

3. Cargas Mensuales vs. Antigüedad (Tenure)
Existe un "punto crítico": clientes con menos de 6 meses de antigüedad y cargos mensuales superiores a 70 USD son el segmento de más alto riesgo.


Conclusión
------------------------
La cancelación no es un evento aleatorio; es un comportamiento predecible. Al integrar el modelo de Random Forest en el sistema de CRM de la empresa, Telecom X puede pasar de una postura reactiva (llamar cuando el cliente ya pidió la baja) a una proactiva (intervenir cuando el modelo detecta un cambio de patrón).

